In [9]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Tabel verb + da andmete kogumine


**Ülesande püstitus**
    
Kõik laused, kus esineb tabelis [list_da.csv](./lists/101.list_da.csv) olev verb ja verbil on otsene alluv deprel=xcomp, feats sisaldab inf

**Tulemus** 

Tabel veergudega:
1. leitud lause, 
2. milline tabelis olevates verbidest seal esineb (algvormis), 
3. da-infinitiivi vormis oleva verbi lemma, 
4. keeletase,
5. emakeel,
6. klass
   

In [10]:
import pandas as pd
from datetime import datetime
from notebook_context import corpus_reader, LISTS_FOLDER

date_time = datetime.now().strftime("%Y%m%d-%H%M%S")


DA_VERBS_LIST =   "./lists/101.list_da.csv"
RESULTS_FILE= LISTS_FOLDER / f"results/da_loend_xcomp_{date_time}.csv"

In [11]:
%%time

# verbid etteantud nimekirjast
df_verbs = pd.read_csv(DA_VERBS_LIST)
my_verbs = list(df_verbs['lemma'].unique())

CPU times: user 2.75 ms, sys: 91 μs, total: 2.84 ms
Wall time: 2.57 ms


In [12]:
my_verbs

['saama',
 'suutma',
 'jaksama',
 'jõudma',
 'nägema',
 'oskama',
 'teadma',
 'mõistma',
 'tohtima',
 'võima',
 'tahtma',
 'kavatsema',
 'plaanima',
 'otsustama',
 'lootma',
 'soovima',
 'igatsema',
 'ihkama',
 'maldama',
 'kärsima',
 'läbema',
 'unistama',
 'ootama',
 'himustama',
 'taotlema',
 'ilgema',
 'sügelema',
 'kibelema',
 'janunema',
 'kaaluma',
 'kavandama',
 'kokku leppima',
 'mõtlema',
 'plaanitsema',
 'planeerima',
 'sihtima',
 'märkama',
 'taipama',
 'unustama',
 'kartma',
 'häbenema',
 'armastama',
 'eelistama',
 'julgema',
 'söandama',
 'tihkama',
 'riskima',
 'usaldama',
 'riskeerima',
 'uskuma',
 'suvatsema',
 'viitsima',
 'paljuks pidama',
 'raatsima',
 'täima',
 'vaevaks võtma',
 'pelgama',
 'põlgama',
 'tõrkuma',
 'kõhklema',
 'pruukima',
 'tarvitsema',
 'lubama',
 'ähvardama',
 'tõotama',
 'vanduma',
 'proovima',
 'püüdma',
 'katsuma',
 'üritama',
 'tavatsema',
 'harrastama',
 'väärima',
 'aitama',
 'ette_panema',
 'hõlbustama',
 'keelama',
 'käskima',
 'laskma',

In [13]:
%%time


collected_data = []
count = 0
for sent_id, graph in corpus_reader.get_sentences():
    # matrix for node distances
    dpath = graph.get_distances_matrix()
    
    # verb nodes
    verb_nodes = [v for v in graph.get_nodes_by_attributes(attrname="POS", attrvalue="VERB") if graph.nodes[v]["lemma"] in my_verbs]
    if not len(verb_nodes): continue
    
    # xcomp
    xcomp_nodes = graph.get_nodes_by_attributes(attrname="deprel", attrvalue="xcomp")
   
    if not len(xcomp_nodes): continue
    
    for verb in verb_nodes:
        # childnodes
        kids = [k for k in dpath[verb] if dpath[verb][k] == 1]
        for xcomp in xcomp_nodes:
            if xcomp not in kids:
                continue
            if not graph.nodes[xcomp]["feats"] or "VerbForm" not in graph.nodes[xcomp]["feats"].keys() or not graph.nodes[xcomp]["feats"]["VerbForm"] == 'Inf':
                continue
            
            #graph.draw_graph2(highlight=[verb, xcomp])
            d = {
                'id':  graph.get_metadata('sent_id'),
                'sentence':  graph.get_metadata('text'),
                'verb':  graph.nodes[verb]["lemma"],
                'xcomp':  graph.nodes[xcomp]["lemma"],
                'sub': " ".join(
                            [graph.nodes[n]["form"] for n in sorted([verb] + kids)]
                        ),
                'keeletase': graph.get_metadata("doc").get("keeletase"),
                'emakeel': graph.get_metadata("doc").get("emakeel"),
                'klass': graph.get_metadata("doc").get("klass")
            }
            
            collected_data.append(d)


../data/vrt-with-meta-corpus-02-06-25_ordered.vrt
CPU times: user 12 s, sys: 116 ms, total: 12.1 s
Wall time: 12.1 s


In [14]:
df = pd.DataFrame.from_dict(collected_data)
df.to_csv(RESULTS_FILE, index=None)
df.head(10)

,id,sentence,verb,xcomp,sub,keeletase,emakeel,klass
0,4095_2,"Selle pärast, et need olid väga ilusad ja taht...",tahtma,lugema,ja tahtis lugeda,0,1,3
1,4097_2,"Keku vaatas taevatähti, et teada saada kui pal...",saama,teadma,", et teada saada palju",0,1,3
2,4112_2,Ta tahtis teada kui palju neid on ja kui ilusa...,tahtma,teadma,Ta tahtis teada .,0,1,3
3,4120_2,"Keku vaatas taevatähti sellepärast, et ta taht...",tahtma,teadma,", et ta tahtis teada ilusad",0,1,3
4,4121_2,"Keku vaata taevatähti, sest ta tahtis neid lug...",tahtma,lugema,", sest ta tahtis lugeda",0,1,3
5,4122_2,Ta tahtis teada palju neid on ja kui ilusad ne...,tahtma,teadma,Ta tahtis teada palju ilusad .,0,1,3
6,4125_2,"Keku vaatas taevatähti, et teada saada kui pal...",saama,teadma,", et teada saada palju",0,1,3
7,4127_2,Keku tahtis teada kui palju neid on ja kui ilu...,tahtma,teadma,Keku tahtis teada .,0,1,3
8,4128_2,Ullu tahtis õpetajale ara kittuda.,tahtma,kittuma,Ullu tahtis kittuda .,0,1,3
9,4131_3,"Keku vaatas taevatähti, sest ta tahtis teada k...",tahtma,teadma,", sest ta tahtis teada",0,1,3
